In [9]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

import h5py
import numpy as np
import pandas as pd
import pickle

from utils import utils_2
from utils import functions as func
from tqdm import tqdm
from utils import parameters as params
import copy
from utils import metabolites as metab
import pickle
import sympy

In [11]:
lp_path = '/data2/hratch/human_me/test_lp/'

# counter = 2
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
    tme = pickle.load(handle)

In [18]:
stat == 1

True

In [13]:
# move to me model class and make #3
mu_val = 1e-9
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln)

Getting MINOS parameters...
Done in 151.747 seconds with status 1


AttributeError: 'float' object has no attribute 'copy'

In [14]:
import copy
def infeasible_reactions(self, mu_val, sln):
    '''
    Should only use for infeasible models to identify reactions that cause infeasibility.
    
    Inputs:
            sln: A solution output from ME_Model.solve_lp
            mu_val: The mu_value that was used in ME_Model.solve_lp
    Ouput: a dictionary with keys as reaction ids and values as fluxes for reactions that caused infeasibility    
    '''
    
    ir = dict()
    for r in self.reactions:
        flux = sln[self.reactions.index(r.id)]
        
        ub = copy.copy(r.upper_bound)
        lb = copy.copy(r.lower_bound)
    
        if isinstance(ub, sympy.Expr):
            ub = float(ub.subs(params.mu, mu_val))
        if isinstance(lb, sympy.Expr):
            lb = float(lb.subs(params.mu, mu_val))
        if (flux > ub) or (flux < lb):
            ir[r.id] = flux
    return ir

# change coefficients

In [37]:
# find maximal value of a coefficient you are replacing in the reaction
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    


In [ ]:
all_fluxes = dict()
key_fluxes = dict()
for biomass_val in [-17, -8, -0.36, -0.25]:
    new_reactions = [r.copy() for r in tqdm(tme.reactions)]
    r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
    r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                     combine = False)
    test_me = func.ME_Model('test')
    test_me.add_reactions(new_reactions)
    print('Begin solve')
    sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
    
    all_fluxes[biomass_val] = dict(zip([r.id for r in tme.reactions], list(sln0[:len(tme.reactions)])))
    key_fluxes[biomass_val] = {r.id: sln0[tme.reactions.index(r.id)] for r in tme.reactions if 'HGNC:12458' in r.id or 'biomass' in r.id}